<a href="https://colab.research.google.com/github/LakshmiKanth11/WEB-SCRAPPING-/blob/main/wikipedia_webscraping_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install Necessary Libraries**

In [1]:
!pip install requests beautifulsoup4


# **Install the library**

In [2]:
!pip install wikipedia-api


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 616.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.2
    Uninstalling click-8.4.2:
      Successfully uninstalled click-8.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.4.1 which is incompatible.


# **Main part of the code**

In [3]:
import wikipediaapi
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from google.colab import userdata # Added import

def get_neat_summary(topic):
    wiki = wikipediaapi.Wikipedia(
        user_agent='ProfessionalScraper/2.0 (lakshmikanth.v07@gmail.com)',
        language='en'
    )
    page = wiki.page(topic)
    if not page.exists():
        return None

    # Get the intro (everything before the first table of contents)
    # This is usually much more than 3 lines
    full_intro = page.summary

    # Split into paragraphs and clean up whitespace
    paragraphs = [p.strip() for p in full_intro.split('\n') if len(p.strip()) > 40]

    return {
        "title": page.title,
        "paragraphs": paragraphs,
        "url": page.fullurl
    }

def send_neat_email(recipient_email, sender_email, app_password, wiki_data):
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = recipient_email
    msg['Subject'] = f"Deep Dive: {wiki_data['title']}"

    # Create a "Neat" HTML version of the email
    # 2025 Standard: Use clean fonts and readable spacing
    paragraphs_html = "".join([f"<p style='line-height: 1.6; font-size: 16px;'>{p}</p>" for p in wiki_data['paragraphs']])

    html_body = f"""
    <html>
        <body style="font-family: Arial, sans-serif; color: #333; padding: 20px;">
            <h2 style="color: #1a0dab; border-bottom: 2px solid #eee; padding-bottom: 10px;">
                {wiki_data['title']}
            </h2>
            {paragraphs_html}
            <br>
            <a href="{wiki_data['url']}"
               style="background-color: #0056b3; color: white; padding: 10px 20px; text-decoration: none; border-radius: 5px; font-weight: bold;">
               Read Full Article on Wikipedia
            </a>
            <p style="font-size: 12px; color: #888; margin-top: 30px;">
                Information retrieved automatically on Dec 24, 2025.
            </p>
        </body>
    </html>
    """

    msg.attach(MIMEText(html_body, 'html'))

    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender_email, app_password)
        server.sendmail(sender_email, recipient_email, msg.as_string())
        server.quit()
        return "Success! Neat summary sent."
    except Exception as e:
        return f"Failed: {e}"

# --- RUN IT ---
MY_GMAIL = "lakshmikanth.v07@gmail.com"
APP_PASS = userdata.get('APP_PASS') # Changed to get from secrets
TARGET_MAIL = "ss903646@gmail.com"
TOPIC = "Cristiano Ronaldo"

data = get_neat_summary(TOPIC)
if data:
    status = send_neat_email(TARGET_MAIL, MY_GMAIL, APP_PASS, data)
    print(status)
else:
    print("Topic not found.")

Success! Neat summary sent.
